In [83]:
# !pip install langchain langchain-core langchain-community pypdf pymupdf sentence-transformers chromadb

In [84]:
from langchain_core.documents import Document

In [85]:
sample_docs = Document(
    page_content="Hello World!",
    metadata={"source":"https//www.google.com"}
)

In [86]:
sample_docs

Document(metadata={'source': 'https//www.google.com'}, page_content='Hello World!')

In [87]:
type(sample_docs)

langchain_core.documents.base.Document

In [88]:
# Text Data..
from langchain_community.document_loaders.text import TextLoader

loader = TextLoader("Data/python.txt",encoding="utf-8")

In [89]:
document = loader.load()

In [90]:
document

[Document(metadata={'source': 'Data/python.txt'}, page_content='\ufeffPython is a high-level, interpreted programming language that has become one of the most popular and widely used languages in the world. Created by Guido van Rossum and first released in 1991, Python emphasizes simplicity and readability, making it easy for beginners to learn while remaining powerful for experienced developers. Its clean and concise syntax allows programmers to write fewer lines of code compared to many other languages, enhancing productivity and maintainability. Python supports multiple programming paradigms, including procedural, object-oriented, and functional programming, which makes it versatile for a wide range of applications.\nSome key features and benefits of Python include:\n* Ease of Learning: Simple syntax and readability make Python beginner-friendly.\n* Versatility: Suitable for web development, data analysis, artificial intelligence, machine learning, scientific computing, automation, 

In [91]:
# PDF Data
from langchain_community.document_loaders.pdf import PyPDFLoader

In [92]:
pdf_loader = PyPDFLoader("data/research1.pdf")

In [93]:
document = pdf_loader.load()
document

[Document(metadata={'producer': 'pdfcpu v0.9.1 dev', 'creator': 'PyPDF', 'creationdate': '2026-03-28T09:36:56+00:00', 'author': 'Ashish Vaswani, Noam Shazeer, Niki Parmar, Jakob Uszkoreit, Llion Jones, Aidan N. Gomez, Łukasz Kaiser, Illia Polosukhin', 'book': 'Advances in Neural Information Processing Systems 30', 'created': '2017', 'date': '2017', 'description': 'Paper accepted and presented at the Neural Information Processing Systems Conference (http://nips.cc/)', 'description-abstract': 'The dominant sequence transduction models are based on complex recurrent orconvolutional neural networks in an encoder and decoder configuration. The best performing such models also connect the encoder and decoder through an attentionm echanisms.  We propose a novel, simple network architecture based solely onan attention mechanism, dispensing with recurrence and convolutions entirely.Experiments on two machine translation tasks show these models to be superiorin quality while being more paralleli

In [94]:
# from langchain_community.document_loaders.pdf import PyMuPDFLoader

# pdf_loader = PyMuPDFLoader("data/research2.pdf")
# document = pdf_loader.load()

In [95]:
# document

# Ingestion Pipeline

## Documents

In [96]:
# data => documents ..
import os
from langchain_community.document_loaders.pdf import PyPDFLoader


In [97]:
def load_all_pdfs():
    folder_path="Data/pdf"
    num_docs=0
    all_docs=[]

    for filename in os.listdir(folder_path):
        if filename.lower().endswith(".pdf"):
            # complete file path
            pdf_path = os.path.join(folder_path,filename)

            loader = PyPDFLoader(pdf_path)
            doc = loader.load()

            all_docs.extend(doc)
            num_docs+=1
    print("total pdfs:",num_docs)
    print("total pages:",len(all_docs))
    return all_docs

In [98]:
all_pdf_documents = load_all_pdfs()

total pdfs: 2
total pages: 32


In [99]:
# chunks 
# !pip install langchain_text_splitters

## Chunks

In [100]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [101]:
def split_docs(documents,chunk_size=500,chunk_overlap=50):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap = chunk_overlap
    )
    chunked_docs = text_splitter.split_documents(documents)
    return chunked_docs

In [102]:
chunks=split_docs(all_pdf_documents)

In [103]:
len(chunks)

321

### Embeddings..

In [104]:
from sentence_transformers import SentenceTransformer

In [105]:
class EmbeddingManager:
    def __init__(self,model_name="all-MiniLM-L6-v2"):
        self.model_name=model_name
        print("Loading model....",self.model_name)
        self.model = SentenceTransformer(self.model_name)
        print("Embedding Dimensions=",self.model.get_sentence_embedding_dimension())

    def generate_embeddings(self,text):
        embeddings = self.model.encode(text,show_progress_bar=True)
        print("embeddings shape:",embeddings.shape)
        return embeddings

In [106]:
embedding_manager = EmbeddingManager()

Loading model.... all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding Dimensions= 384


C:\Users\lenovo\AppData\Local\Temp\ipykernel_15884\2735686109.py:6: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("Embedding Dimensions=",self.model.get_sentence_embedding_dimension())


### Vector Store..

In [107]:
import chromadb
import uuid

In [108]:
class VectorStoreManager:
    def __init__(self,persist_directory="data/vector_store",collection_name="pdf_documents"): # collection name is a table name where our embeddings store..
        self.collection_name=collection_name
        self.persist_directory=persist_directory
        self.collection = None
        self.client = None

        self._initialize_store()

    def _initialize_store(self):
        os.makedirs(self.persist_directory,exist_ok=True)
        ## create a client..
        self.client = chromadb.PersistentClient(path=self.persist_directory)

        ## create the collection..
        self.collection = self.client.get_or_create_collection(
            name=self.collection_name,
            metadata={"description":"vector store collection for pdf embeddings in RAG"}
        )
        print("Initialize the vector store with collection:",self.collection_name)
        print("docs in collection:",self.collection.count())

    def add_documents(self, documents,embeddings): ## here,we use it to store embeddings and documents to store in vector db
        if len(documents) != len(embeddings):
            raise ValueError("num of documents does not match with num of embeddings")

        # store => ids,embeddings,documents and metadata..
        ids = []
        all_metadata =[]
        documents_content=[]
        embedding_list=[]

        for i,(doc,embedding) in enumerate(zip(documents,embeddings)):
            doc_id = f"doc_{uuid.uuid4()}" # doc_a1,doc_a2
            ids.append(doc_id)

            metadata= dict(doc.metadata)
            metadata["doc_index"]=i
            metadata["content_length"]=len(doc.page_content)
            all_metadata.append(metadata)

            documents_content.append(doc.page_content)
            
            embedding_list.append(embedding.tolist())

            self.collection.add(
                ids=ids,
                metadatas=all_metadata,
                documents = documents_content,
                embeddings=embedding_list
            )

        print("total documents added in vector store=",len(documents_content))
        print("docs in collection:",self.collection.count())

        
            

In [109]:
vector_store = VectorStoreManager()

Initialize the vector store with collection: pdf_documents
docs in collection: 642


In [110]:
# data ==> documents => chunks =>embeddings =>store in vector store
texts = [doc.page_content for doc in chunks]

embedding=embedding_manager.generate_embeddings(texts)

vector_store.add_documents(chunks,embedding)

Batches:   0%|          | 0/11 [00:00<?, ?it/s]

embeddings shape: (321, 384)
total documents added in vector store= 321
docs in collection: 963


#### Reterival pipeline

In [111]:
from sklearn.metrics.pairwise import cosine_similarity

In [130]:
class RAGRetriever:
    def __init__(self,embedding_manager,vector_store):
        self.embedding_manager = embedding_manager
        self.vector_store = vector_store

    def retriever(self,query,top_k=5,score_threshold=0.0):
        # query => embedding.
        query_embeddings = self.embedding_manager.generate_embeddings([query])[0]

        # semantic search..
        results=self.vector_store.collection.query(
            query_embeddings = [query_embeddings.tolist()],
            n_results=top_k
        )

        # cosine similarity..
        retrieved_docs =[]
        if results["documents"] and results["documents"][0]:
            ids = results["ids"][0]
            metadatas = results["metadatas"][0]
            documents = results["documents"][0]
            distances = results["distances"][0]

            for i, (doc_id,metadata,document,distance) in enumerate(zip(ids,metadatas,documents,distances)):
                similarity_score = 1-distance

                if similarity_score >= score_threshold:
                    retrieved_docs.append({
                        "id":doc_id,
                        "document":document,
                        "metadata":metadata,
                        "distance":distance,
                        "similarity_score":similarity_score,
                        "rank":i+1
                    })
            print(f"retrived {len(retrieved_docs)} documents")
            

        else:
            print("no documents found")

        return retrieved_docs ## context..
        

In [131]:
rag_retriever = RAGRetriever(embedding_manager,vector_store)

In [132]:
rag_retriever.retriever("What is RAG?")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

embeddings shape: (1, 384)
retrived 5 documents


[{'id': 'doc_eb98ee41-3df5-4837-8eac-3973db160bd1',
  'document': 'and speculate on upcoming trends and innovations.\nOur contributions are as follows:\n• In this survey, we present a thorough and systematic\nreview of the state-of-the-art RAG methods, delineating\nits evolution through paradigms including naive RAG,\narXiv:2312.10997v5  [cs.CL]  27 Mar 2024',
  'metadata': {'source': 'Data/pdf\\research2.pdf',
   'author': '',
   'subject': '',
   'page': 0,
   'content_length': 288,
   'creationdate': '2024-03-28T00:54:45+00:00',
   'keywords': '',
   'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5',
   'producer': 'pdfTeX-1.40.25',
   'page_label': '1',
   'moddate': '2024-03-28T00:54:45+00:00',
   'trapped': '/False',
   'doc_index': 88,
   'title': '',
   'creator': 'LaTeX with hyperref',
   'total_pages': 21},
  'distance': 0.4629147946834564,
  'similarity_score': 0.5370852053165436,
  'rank': 1},
 {'id': 'doc_77076926-f

# Intrigate RAG with LLMs

## OpenAI - GPT

In [139]:
API_KEY_OPENAI=""

In [134]:
# !pip install langchain-openai

In [141]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    open_api_key = API_KEY_OPENAI,
    model= "gpt-5.4",
    temperature=0.1,
    max_tokens=1024
)


In [149]:
# generate our retirever-augmented output..
def generate_output(query,retriever,llm,top_k=3):
    results = retriever.retriever(query,top_k)

    context= "\n".join(doc["document"] for doc in results) if results else ""

    if not context:
        print("we found no relavent context for the given query")
    # prompt = context+query
    prompt = f""" use given context to generate the answer
             Context: {context}
             Query: {query} """

    response = llm.invoke(prompt) # expecting string as prompt..
    return response.content

In [150]:
answer = generate_output("what is RAG?",rag_retriever,llm)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

embeddings shape: (1, 384)
retrived 3 documents


RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}

In [152]:
print(answer)

NameError: name 'answer' is not defined

### GROQ 

In [157]:
API_Key_GROQ=""

In [158]:
# !pip install langchain-groq

In [163]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    groq_api_key = API_Key_GROQ,
    model="qwen/qwen3-32b",
    temperature=0.1,
    max_tokens=1024
)


In [164]:
# generate our retirever-augmented output..
def generate_output(query,retriever,llm,top_k=3):
    results = retriever.retriever(query,top_k)

    context= "\n".join(doc["document"] for doc in results) if results else ""

    if not context:
        print("we found no relavent context for the given query")
    # prompt = context+query
    prompt = f""" use given context to generate the answer
             Context: {context}
             Query: {query} """

    response = llm.invoke([prompt.format(context=context,query=query)]) # expecting string as prompt..
    return response.content

In [165]:
answer = generate_output("what is RAG?",rag_retriever,llm)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

embeddings shape: (1, 384)
retrived 3 documents


In [166]:
print(answer)

<think>
Okay, the user is asking, "What is RAG?" Let me start by recalling what I know about RAG. From the context provided, there's a mention of a survey that reviews state-of-the-art RAG methods and discusses their evolution through paradigms like naive RAG. The arXiv paper is cited multiple times, so that's probably a key source.

First, I need to define RAG. RAG stands for Retrieval-Augmented Generation. It's a technique that combines retrieval and generation models. The basic idea is that instead of relying solely on the model's internal knowledge, it retrieves relevant information from an external source and then uses that to generate a response. This helps in providing more accurate and up-to-date information, especially for factual queries.

Looking at the context, the survey mentions different paradigms like naive RAG. I should explain what naive RAG is. Naive RAG might be the simplest form where the retrieval and generation steps are done sequentially without much optimizatio